## Surface Plot Tests

by: Brett Mattas
<br>date: 5/17/2016

### Purpose

The purpose of this notebook is to generate plotly Surface plots in a manner I'll be using throughout this library.
[Surface Plots](https://plotly.com/python/3d-surface-plots/)

### Included:

1. Basic surface plot with some formatting explored. These are useful for taking a slice of fill and seeing how pressures vary over the area of the slice.
2. How to generate a surfaces at multiple slices. I use this for keeping the same x & y coordinates, but taking different slices varying the z coordinate.
3. Comparing different surface plots at the same locations.

In [1]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from math import pi
from buried_structures import Point_3d, Origin
import xarray as xr
from ipywidgets import Output
import plotly.io as pio
from IPython.display import display, HTML
print("Import tests finished")

Import tests finished


In [2]:
def bous(p: Point_3d) -> float:
    # Simplified Boussinesq problem with hard coded
    # inputs that only returns downward pressure.
    # Used for demonstrating plotly charts work as
    # intended.

    o = Origin()
    R = p.distance(o)
    R = R if R > 0 else 0.001 # Avoid singularities
    z = p.dz(o)
    P = 1.0

    # print(f"{R=}, {z=}")
    return 3*P*(z**3) / (2 * pi * (R**5))

p = Point_3d(1,1,1)
print(f"pressure = {bous(p)}")

pressure = 0.030629383078988458


In [3]:
def find_extreme(iterable, func=max) -> float|None:
    """
    Recursively find the min or max in nested iterables.
    :param iterable: The nested collection (list, tuple, etc.)
    :param func: The built-in min or max function
    """
    flat_elements = []
    
    for item in iterable:
        # Check if the item is a nested iterable (excluding strings if desired)
        if isinstance(item, (list, tuple, set)):
            # Recursive call to drill into the nested structure
            inner_extreme = find_extreme(item, func)
            if inner_extreme is not None:
                flat_elements.append(inner_extreme)
        else:
            # Base case: item is a single value (int, float, etc.)
            flat_elements.append(item)
            
    return func(flat_elements) if flat_elements else None

# Example usage:
nested_data = [1, [5, [10, -2]], 8, [0]]
print(f"Maximum: {find_extreme(nested_data, max)}") # Output: 10
print(f"Minimum: {find_extreme(nested_data, min)}") # Output: -2


Maximum: 10
Minimum: -2


In [4]:
# Single Surface Plot

x, y = np.linspace(0, 5, 10), np.linspace(0, 5, 10)

z_coordinate = 10
z = []

for ix in x:
    new_row = []

    for iy in y:
        point = Point_3d(x=ix, y=iy, z=z_coordinate)
        new_row.append(bous(point))
    z.append(new_row)

fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale="Turbo", 
                                 colorbar=dict(title="Title"))])

# Some formatting
fig.update_layout(title=f"Boussinesq Pressure at z={z_coordinate} in",
                  width = 800, height=600,
                  scene=dict(
                    xaxis_title="x (in)",
                    yaxis_title = "y (in)",
                    zaxis_title = "Pressure (ksi)"
                    )
                  )

fig.show()

In [5]:
# Multiple Surface Plots on top of each other

x, y = np.linspace(0, 5, 10), np.linspace(0, 5, 10)
z_coordinate = np.linspace(10, 12, 3)

z_all = []

for iz in z_coordinate:
    z = []

    for ix in x:
        new_row = []

        for iy in y:
            point = Point_3d(x=ix, y=iy, z=iz)
            pressure = bous(point)
            new_row.append(bous(point))
        z.append(new_row)

    z_all.append(z)

cmin = find_extreme(z_all, min)
cmax = find_extreme(z_all, max)

fig = go.Figure()

for iz, z_c in enumerate(z_coordinate):
    show_colorbar = True if iz == 0 else False
    fig.add_trace(go.Surface(z=z_all[iz], x=x, y=y, 
                            colorscale="Turbo", opacity=0.5, 
                            name=f"z={z_c}", 
                            cmin=cmin, cmax=cmax,
                            showscale=show_colorbar
                            )
                    )

# Some formatting
z_string = "10, 11, 12"
fig.update_layout(title=f"Boussinesq Pressure at z={z_string} in",
                  width = 800, height=600,
                  scene=dict(
                    xaxis_title="x (in)",
                    yaxis_title = "y (in)",
                    zaxis_title = "Pressure (ksi)"
                    ),
                  )
print(f"{cmin=}, {cmax=}")
fig.show()

cmin=np.float64(0.0015739132796344973), cmax=np.float64(0.004774648292756861)


In [6]:
fig_solid = go.Figure()
print(z_coordinate)
solid_colors = ["red", "green", "blue"]
for i, color in enumerate(solid_colors):
    fig_solid.add_trace(go.Surface(
        z=z_all[i],
        x=x, y=y,
        surfacecolor=np.zeros((len(x), len(y))),
        colorscale=[[0, color], [1, color]],
        opacity=0.5,
        name=f"z={z_coordinate[i]:.0f}",
        showscale=False
    ))

fig_solid.update_layout(
    title="Boussinesq Pressure at z=10, 11, 12 in (solid colors)",
    width=800,
    height=600,
    scene=dict(
        xaxis_title="x (in)",
        yaxis_title="y (in)",
        zaxis_title="Pressure (ksi)"
    )
)

fig_solid.show()

[10. 11. 12.]


In [7]:
# Change the Up direction for horizontal pressures.

x, y = np.linspace(0, 5, 10), np.linspace(0, 5, 10)

z_coordinate = 10
z = []

for ix in x:
    new_row = []

    for iy in y:
        point = Point_3d(x=ix, y=iy, z=z_coordinate)
        new_row.append(bous(point))
    z.append(new_row)

fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale="Turbo", 
                                 colorbar=dict(title="Title"))])

# Some formatting
zoom = 0.3
z_max = find_extreme(z, max)
z_max = 0.01 if z_max is None else z_max

camera = dict(
    up=dict(x=0, y=1, z=0),
    center=dict(x=0,y=0,z=0),
    eye=dict(x=zoom*max(x), y=zoom*max(y),z=zoom*max(x))
)

camera_override = dict(
    eye=dict(x=1.46, y=1.15, z=1.22),
    center=dict(x=0.19, y=-0.13, z=-0.06),
    up=dict(x=-0.41, y=0.82, z=-0.41)
)

fig.update_layout(title=f"Boussinesq Pressure at z={z_coordinate} in",
                  width = 800, height=600,
                  scene=dict(
                      camera=camera_override,
                      xaxis_title="x (in)",
                      yaxis_title = "y (in)",
                      zaxis_title = "Pressure (ksi)",
                      yaxis=dict(autorange="reversed")
                    )
                  )
config = {'displayModeBar': True}

fig.show(config=config)

In [8]:
# Get camera output version

camera_override = dict(
    eye=dict(x=1.46, y=1.15, z=1.22),
    center=dict(x=0.19, y=-0.13, z=-0.06),
    up=dict(x=-0.41, y=0.82, z=-0.41)
)
fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale="Turbo", 
                                 colorbar=dict(title="Title"))])
fig.update_layout(title=f"Boussinesq Pressure at z={z_coordinate} in",
                  width = 800, height=600,
                  scene=dict(
                      camera=camera_override,
                      xaxis_title="x (in)",
                      yaxis_title = "y (in)",
                      zaxis_title = "Pressure (ksi)",
                      yaxis=dict(autorange="reversed")
                    )
                  )
config = {'displayModeBar': True}
fig.show()


# 4. Inject an HTML text container AND the JavaScript listener block
# This finds your latest plot, extracts layout events, and rewrites the <pre> tag text.
display(HTML('''
<div style="margin-top: 15px; font-family: monospace;">
    <h4 style="margin-bottom: 5px; color: #444;">Live Camera Tracker Output:</h4>
    <pre id="camera-output" style="
        background: #f4f4f5; 
        padding: 12px; 
        border: 1px solid #e4e4e7; 
        border-radius: 6px; 
        color: #18181b;
        font-size: 13px;
        line-height: 1.5;
        max-width: 600px;
    ">Rotate or zoom the chart above to extract Python layout code...</pre>
</div>

<script>
setTimeout(() => {
    // Locate the Plotly graph wrapper generated in this cell
    var plots = document.getElementsByClassName('js-plotly-plot');
    if (plots.length > 0) {
        var gd = plots[plots.length - 1]; 
        var outputContainer = document.getElementById('camera-output');
        
        // Listen to the native layout change event
        gd.on('plotly_relayout', function(eventData) {
            if (eventData && eventData['scene.camera']) {
                var cam = eventData['scene.camera'];
                
                // Construct the clean, copy-pasteable configuration block
                var codeString = `scene_camera = dict(\\n` +
                                 `    eye=dict(x=${cam.eye.x.toFixed(2)}, y=${cam.eye.y.toFixed(2)}, z=${cam.eye.z.toFixed(2)}),\\n` +
                                 `    center=dict(x=${cam.center.x.toFixed(2)}, y=${cam.center.y.toFixed(2)}, z=${cam.center.z.toFixed(2)}),\\n` +
                                 `    up=dict(x=${cam.up.x.toFixed(2)}, y=${cam.up.y.toFixed(2)}, z=${cam.up.z.toFixed(2)})\\n` +
                                 `)`;
                
                // Safely update the inline text area element
                outputContainer.innerText = codeString;
            }
        });
    }
}, 1000);
</script>
'''))

In [9]:
# Stacked flat surfaces colored by Boussinesq pressure (z = 10, 11, 12)
z_coords = [10.0, 11.0, 12.0]

pressure_mats = []
flat_z_mats = []
for zc in z_coords:
    p_mat = []
    z_mat = []
    for ix in x:
        p_row = []
        z_row = []
        for iy in y:
            p_row.append(bous(Point_3d(x=ix, y=iy, z=zc)))
            z_row.append(zc)
        p_mat.append(p_row)
        z_mat.append(z_row)
    pressure_mats.append(p_mat)
    flat_z_mats.append(z_mat)

fig_stacked = go.Figure()
for i, zc in enumerate(z_coords):
    fig_stacked.add_trace(go.Surface(
        z=flat_z_mats[i],
        x=x, y=y,
        surfacecolor=pressure_mats[i],
        colorscale="Jet",
        cmin=cmin, cmax=cmax,
        opacity=0.5,
        name=f"z={zc}",
        showscale=(i == 0),
        colorbar=(dict(title="Pressure (ksi)")) if (i == 0) else None
    ))

fig_stacked.update_layout(title="Stacked pressures",
                          width=1000, height=600,
                          scene=dict(
                              xaxis_title="x (in)",
                              yaxis_title="y (in)",
                              zaxis_title="z (in)",
                              zaxis=dict(autorange="reversed")
                            ),
                          )

fig_stacked.show()